In [113]:
# !pip install ucimlrepo

In [114]:
import random
import numpy as np
from pygrad.optimizers.sgd import SGD
from pygrad.optimizers import backprop
from pygrad.tensor.tensor import Tensor
from ucimlrepo import fetch_ucirepo 

In [115]:
# fetch dataset 
iris = fetch_ucirepo(id=53) 
  
# data (as pandas dataframes) 
X = iris.data.features 
y = iris.data.targets 
  
# metadata 
print(iris.metadata) 
  
# variable information 
print(iris.variables) 

{'uci_id': 53, 'name': 'Iris', 'repository_url': 'https://archive.ics.uci.edu/dataset/53/iris', 'data_url': 'https://archive.ics.uci.edu/static/public/53/data.csv', 'abstract': 'A small classic dataset from Fisher, 1936. One of the earliest known datasets used for evaluating classification methods.\n', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Tabular'], 'num_instances': 150, 'num_features': 4, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1936, 'last_updated': 'Tue Sep 12 2023', 'dataset_doi': '10.24432/C56C76', 'creators': ['R. A. Fisher'], 'intro_paper': {'ID': 191, 'type': 'NATIVE', 'title': 'The Iris data set: In search of the source of virginica', 'authors': 'A. Unwin, K. Kleinman', 'venue': 'Significance, 2021', 'year': 2021, 'journal': 'Significance, 2021', 'DOI': '1740-9713.01589', 'URL': 'https://www.semanticscholar.org

In [116]:
X = X.to_numpy()
y = y.to_numpy()

In [117]:
indices = np.random.permutation(len(X))
X = X[indices]
y = y[indices]

In [118]:
X_test = X[:10]
Y_test = y[:10]

In [119]:
X = X[10:]
y = y[10:]

In [120]:
y = np.where(y == 'Iris-setosa', 1, 0)

In [121]:
W_1 = Tensor(
    np.random.randn(len(X[0]), 1) * 0.1,
    device="cuda",
    requires_grad=True,
)
W_1

Tensor([[0.12429462]
 [0.06217644]
 [0.18180151]
 [0.10944227]]) (0x7f7ba394cd40))

In [122]:
b_1 = Tensor(0, device="cuda", requires_grad=True)
b_1

Tensor(0.0) (0x7f7ba39d3d10))

In [123]:
def forward(X, W_1, b_1):
    h_1 = X@W_1 + b_1
    a_1 = h_1.sigmoid()
    
    return a_1

In [124]:
trainble = (W_1, b_1)

In [125]:
sdg = SGD(trainble, epcilon=3e-3)

In [ ]:
for j in range(100):
    for i in range(0, len(X), 10):
        sdg.zero_grad()
        if i+10 >= len(X): 
            print("Dataset finished")
            break
        X_batch = Tensor(X[i:i+10], device='cuda')
        y_batch = Tensor(y[i:i+10], device='cuda')
        
        y_pred = forward(X_batch, W_1, b_1)
        
        loss = ((y_pred - y_batch) ** 2).mean()
        print(f"Iteration: {i/10}, loss: {loss}")
        backprop.backward(loss)
        sdg.step()

Iteration: 0.0, loss: Tensor(0.5751374959945679)
Iteration: 1.0, loss: Tensor(0.5617421865463257)
Iteration: 2.0, loss: Tensor(0.4787355363368988)
Iteration: 3.0, loss: Tensor(0.48152393102645874)
Iteration: 4.0, loss: Tensor(0.4170842170715332)
Iteration: 5.0, loss: Tensor(0.5624555945396423)
Iteration: 6.0, loss: Tensor(0.4770023226737976)
Iteration: 7.0, loss: Tensor(0.6163197159767151)
Iteration: 8.0, loss: Tensor(0.48154324293136597)
Iteration: 9.0, loss: Tensor(0.599524974822998)
Iteration: 10.0, loss: Tensor(0.46957820653915405)
Iteration: 11.0, loss: Tensor(0.4740681052207947)
Iteration: 12.0, loss: Tensor(0.5820372104644775)
Dataset finished
Iteration: 0.0, loss: Tensor(0.5392411947250366)
Iteration: 1.0, loss: Tensor(0.52446049451828)
Iteration: 2.0, loss: Tensor(0.4484183192253113)
Iteration: 3.0, loss: Tensor(0.4500296711921692)
Iteration: 4.0, loss: Tensor(0.39434897899627686)
Iteration: 5.0, loss: Tensor(0.5211944580078125)
Iteration: 6.0, loss: Tensor(0.44342508912086487

In [127]:
X_test = Tensor(X_test, device='cuda')

In [128]:
y_pred_test = forward(X_test, W_1, b_1)

In [129]:
np.where(y_pred_test.data>=0.5, 1, 0)

array([[1],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0]])

In [130]:
Y_test

array([['Iris-setosa'],
       ['Iris-versicolor'],
       ['Iris-virginica'],
       ['Iris-virginica'],
       ['Iris-versicolor'],
       ['Iris-virginica'],
       ['Iris-versicolor'],
       ['Iris-setosa'],
       ['Iris-virginica'],
       ['Iris-versicolor']], dtype=object)